In [ ]:
## Generating spectogram images for files used in training CoAtNet

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, ToTensor
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt


In [21]:
# waveform function for me to not bang my keyboard
def disp_waveform(signal, sr=None):
    plt.figure(figsize=(7,2))
    return librosa.display.waveshow(signal, sr=sr)

In [22]:
def isolator(signal, sample_rate, n_fft, hop_length, before, after, threshold=None, show=False,
             band_fmin=None, band_fmax=None):
    strokes = []
    stroke_times = []
    # -- signal'
    if show:
        disp_waveform(signal, sr=sample_rate)
    fft = librosa.stft(signal, n_fft=n_fft, hop_length=hop_length)

    # band-limited energy (optional)
    if band_fmin is not None or band_fmax is not None:
        freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=n_fft)
        if band_fmin is None:
            band_fmin = 1000#2000
        if band_fmax is None:
            band_fmax = 20000
        band_mask = (freqs >= band_fmin) & (freqs <= band_fmax)
        energy = np.sum(np.abs(fft[band_mask])**2, axis=0)
    else:
        energy = np.sum(np.abs(fft)**2, axis=0)   # power 합 (표준)

    # threshold auto from same signal unless provided
    if threshold is None:
        threshold = np.percentile(energy, threshold_percentile)
    # -- energy'
    if show:
        disp_waveform(energy)
    threshed = energy > threshold
    # -- peaks'
    if show:
        disp_waveform(threshed.astype(float))
    peaks = np.where(threshed == True)[0]
    peak_count = len(peaks)
    prev_end = sample_rate*min_keystroke_gap*(-1)
    # '-- isolating keystrokes'
    for i in range(peak_count):
        this_peak = peaks[i]
        timestamp = (this_peak*hop_length) + n_fft//2
        if timestamp > prev_end + (min_keystroke_gap*sample_rate):
            keystroke = signal[timestamp-before:timestamp+after]
            strokes.append(keystroke)
            stroke_times.append(timestamp / sample_rate)
            if show:
                disp_waveform(keystroke, sr=sample_rate)
            prev_end = timestamp+after
    print("peaks:", len(peaks))
    print("strokes:", len(strokes))
    
    return strokes, stroke_times


In [23]:
# Unified parameters (run this once)
n_fft = 1024
hop_length = 100#256
n_mels = 64
mel_n_fft = 1024
mel_hop_length = 225
mel_fmin = 0
mel_fmax = 19000
before = 2400
after = 3000

# threshold percentile used inside isolator when threshold=None
threshold_percentile = 95

# 최소 키 누르기 간격 (초 단위) - 동시 타이핑 고려
# for CoAtNet training dataset, set this to 0.5 ~ 1.0
# for GNN training dataset, set this to 0.03 ~ 0.1
min_keystroke_gap = 0.03 # 50ms (손가락 2개 겹침 허용), 필요시 0.03~0.1로 조정

# band-limited energy for keystroke detection
band_fmin = 1000#2000
band_fmax = 20000


In [24]:
def create_dataset(n_fft, hop_length, before, after):
    for i, File in enumerate(keys):
        loc = MBP_AUDIO_DIR + File
        samples, sr = librosa.load(loc, sr=40000)
        # samples, sr = librosa.load(loc,sr=None,duration=1.0,mono=True)

        strokes = []
        thr = np.percentile(energy, 97) 
        step = 0.005
        strokes = isolator(samples[1*sr:], sr, n_fft, hop_length, before, after, thr, False, band_fmin=band_fmin, band_fmax=band_fmax )
        # while not len(strokes) == 25:
        #   strokes = isolator(samples[1*sr:], sr, n_fft, hop_length, before, after, prom, False )
        #   if len(strokes) < 25:
        #     prom -= step
        #   if len(strokes) > 25:
        #     prom += step
        #   if prom < 0:
        #     print("--not possible for : ", File)
        #     break
        #   step = step * 0.99
        label = [labels[i]]*len(strokes)
        data_dict['Key'] += label
        data_dict['File'] += strokes

    df = pd.DataFrame(data_dict)
    mapper = {}
    counter = 0
    for l in df['Key']:
        if not l in mapper:
            mapper[l] = counter
            counter += 1
    df.replace({'Key': mapper}, inplace = True)

    return df

## Generating MelSpectrograms

In [25]:
# --- Feature extractor setup ---
import numpy as np
import torch
from feature_extraction import CoAtNet

# If you have pretrained weights, set path here; otherwise it will use random weights.
weights_path = None  # e.g., 'coatnet_weights.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CoAtNet().to(device)
if weights_path:
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
model.eval()


CoAtNet(
  (s0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (s1): MBConv(
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (proj): Conv2d(64, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (conv): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=False)
      (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): SE(
        (avg_pool): AdaptiveAvgPool2d(output_size=1)
        (fc): Sequential(
          (0): Linear(in_featu

In [26]:
# --- Mel -> 768 feature ---
import librosa
import librosa.display
import torch.nn.functional as F

@torch.no_grad()
def extract_feature_from_wave(wave, sr):
    mel = librosa.feature.melspectrogram(y=wave, sr=sr, n_mels=n_mels, n_fft=mel_n_fft, hop_length=mel_hop_length)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # normalize to 0..1
    mel_min, mel_max = mel_db.min(), mel_db.max()
    mel_norm = (mel_db - mel_min) / (mel_max - mel_min + 1e-8)

    # to 3-channel tensor [1,3,H,W]
    img = np.stack([mel_norm, mel_norm, mel_norm], axis=0)  # [3,H,W]
    img_t = torch.from_numpy(img).unsqueeze(0).float().to(device)

    # resize to 224x224
    img_t = F.interpolate(img_t, size=(224, 224), mode='bilinear', align_corners=False)

    # normalize (ImageNet stats)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    img_t = (img_t - mean) / std

    feat = model(img_t).squeeze(0).cpu().numpy()  # [768]
    return feat


In [27]:
# --- Process wav files: strokes -> mel images -> NPZ ---
from pathlib import Path
import matplotlib.pyplot as plt

test_dir = Path('wav/test')
out_root = Path('out/test')
out_root.mkdir(parents=True, exist_ok=True)

npz_dir = Path('npz_sequences')
npz_dir.mkdir(parents=True, exist_ok=True)


for wav_path in sorted(test_dir.rglob('*.wav')):
    rel = wav_path.relative_to(test_dir)
    rel_dir = rel.parent
    key = wav_path.stem
    samples, sr = librosa.load(wav_path.as_posix(), sr=40000)

    print('file:', wav_path.name)
    print('sr:', sr, 'len:', len(samples))
    print('n_fft:', n_fft, 'hop:', hop_length, 'before:', before, 'after:', after)

    # isolate keystrokes with auto-threshold on the same signal
    strokes, stroke_times = isolator(samples, sr, n_fft, hop_length, before, after, threshold=None, show=False,
                                     band_fmin=band_fmin, band_fmax=band_fmax)

    # save mel images
    key_out = out_root / rel_dir / key
    key_out.mkdir(parents=True, exist_ok=True)
    for i, stroke in enumerate(strokes):
        mel = librosa.feature.melspectrogram(
            y=np.asarray(stroke).squeeze(),
            sr=sr,
            n_mels=n_mels,
            n_fft=mel_n_fft,
            hop_length=mel_hop_length,
            fmin=mel_fmin,
            fmax=mel_fmax,
        )
        plt.figure(figsize=(6, 4))
        librosa.display.specshow(
            librosa.power_to_db(mel, ref=np.max),
            x_axis='time',
            y_axis='mel',
            sr=sr,
            hop_length=mel_hop_length,
        )
        plt.colorbar(format='%+2.0f dB')
        plt.title(f'Mel Spectrogram ({key}) - keystroke {i}')
        plt.tight_layout()
        plt.savefig(key_out / f'keystroke_{i:04d}.png', dpi=150)
        plt.close()

    # build IKI (seconds) and features
    if len(stroke_times) >= 2:
        iki = np.diff(np.array(stroke_times, dtype=float))
    else:
        iki = np.array([], dtype=float)

    features = []
    for stroke in strokes:
        feat = extract_feature_from_wave(np.asarray(stroke).squeeze(), sr)
        features.append(feat)

    x = np.stack(features, axis=0) if features else np.zeros((0, 768), dtype=float)

    # label string from filename: key_<label>.wav
    label_str = key
    if label_str.startswith('key_'):
        label_str = label_str[len('key_'):]

    out_npz = npz_dir / f'key_{label_str}.npz'
    np.savez(out_npz, x=x, iki=iki)
    print('Saved npz:', out_npz)


file: HelloWorld1.wav
sr: 40000 len: 132267
n_fft: 1024 hop: 100 before: 2400 after: 3000
peaks: 67
strokes: 12
Saved npz: npz_sequences/key_HelloWorld1.npz
file: HelloWorlds2.wav
sr: 40000 len: 133974
n_fft: 1024 hop: 100 before: 2400 after: 3000
peaks: 67
strokes: 10
Saved npz: npz_sequences/key_HelloWorlds2.npz
file: HelloWorlds3.wav
sr: 40000 len: 153600
n_fft: 1024 hop: 100 before: 2400 after: 3000
peaks: 77
strokes: 11
Saved npz: npz_sequences/key_HelloWorlds3.npz
file: HelloWorlds4.wav
sr: 40000 len: 183467
n_fft: 1024 hop: 100 before: 2400 after: 3000
peaks: 92
strokes: 13
Saved npz: npz_sequences/key_HelloWorlds4.npz
file: 음성 260220_015631.wav
sr: 40000 len: 124587
n_fft: 1024 hop: 100 before: 2400 after: 3000
peaks: 63
strokes: 10


/var/folders/mz/2wl7n1990lv2x93xqyw579x80000gn/T/ipykernel_7521/1893686161.py:50: UserWarning: Glyph 51020 (\N{HANGUL SYLLABLE EUM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/mz/2wl7n1990lv2x93xqyw579x80000gn/T/ipykernel_7521/1893686161.py:50: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/mz/2wl7n1990lv2x93xqyw579x80000gn/T/ipykernel_7521/1893686161.py:51: UserWarning: Glyph 51020 (\N{HANGUL SYLLABLE EUM}) missing from font(s) DejaVu Sans.
  plt.savefig(key_out / f'keystroke_{i:04d}.png', dpi=150)
/var/folders/mz/2wl7n1990lv2x93xqyw579x80000gn/T/ipykernel_7521/1893686161.py:51: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  plt.savefig(key_out / f'keystroke_{i:04d}.png', dpi=150)


Saved npz: npz_sequences/key_음성 260220_015631.npz
